In [5]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

In [6]:
spark = SparkSession.builder.appName("Phase1").getOrCreate() #This intializes a SparkSession, it runs on the driver node and is the entry point to programming Spark with the Dataset and DataFrame API. It allows you to create DataFrames, register DataFrames as tables, execute SQL over tables, cache tables, and read parquet files. It also provides a way to configure Spark and manage the underlying resources.

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/23 09:05:51 WARN Utils: Your hostname, LAPTOP-H1TA7CV3, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/06/23 09:05:51 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/23 09:05:52 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [7]:
# Dummy Dataset: A small list of knights at the tourney
tourney_data = [
    ("Ser Duncan the Tall", "Crownlands", 5),
    ("Prince Baelor Breakspear", "Crownlands", 1000),
    ("Prince Aerion Brightflame", "Crownlands", 800),
    ("Ser Lyonel Baratheon", "Stormlands", 400),
    ("Ser Robyn Rhysling", "The Reach", 50)
]

columns = ["knight_name", "region", "gold_dragons"]
df_knights = spark.createDataFrame(tourney_data, columns)

In [8]:
# --- NARROW TRANSFORMATION ---
# Lazy Evaluation: No execution happens. The DAG simply records a filter step.
# No data moves across the network. 
df_hedge_knights = df_knights.filter(col("gold_dragons") < 100)

In [9]:
# --- WIDE TRANSFORMATION ---
# Lazy Evaluation: No execution yet, BUT the DAG plans a Shuffle.
# Data will eventually need to be moved across executors so knights of the same 
# region end up on the same machine to be summed up.
df_regional_wealth = df_knights.groupBy("region").sum("gold_dragons")

In [10]:
# --- ACTION ---
# This is the horn blowing! The Driver sends the DAG to the Executors.
# The data is read, filtered, and then returned to the console.
print("Knights of modest means (Hedge Knights):")
df_hedge_knights.show()

Knights of modest means (Hedge Knights):


+-------------------+----------+------------+
|        knight_name|    region|gold_dragons|
+-------------------+----------+------------+
|Ser Duncan the Tall|Crownlands|           5|
| Ser Robyn Rhysling| The Reach|          50|
+-------------------+----------+------------+



In [11]:
# Another Action triggering the wide transformation plan.
print("Total wealth by region at the tourney:")
df_regional_wealth.show()

Total wealth by region at the tourney:


+----------+-----------------+
|    region|sum(gold_dragons)|
+----------+-----------------+
|Crownlands|             1805|
|Stormlands|              400|
| The Reach|               50|
+----------+-----------------+

